In [ ]:
## # What’s the internet usage policy?

### What’s the internet usage policy?

### Step 1: Define Sample Documents

In [3]:
documents = [
    {"doc_id": "1", "section": "Pay Policies", "content": "Employees are paid bi-weekly via direct deposit."},
    {"section": "Leave of Absence", "content": "Employees must submit a leave request for approval."},
    {"section": "Internet Use", "content": "Company internet must be used for work-related tasks only."},
    {"section": "Internet Use", "content": "Company internet is a broadband internet."},
    {"section": "Break at Work", "content": "Employees can take an hour break."},
    {"section": "Harassment", "content": "Interact with each employee with Respect"}
]

## Step 2: Get Content Texts


In [4]:
content_corpus= [doc['content'] for doc in documents]
content_corpus

['Employees are paid bi-weekly via direct deposit.',
 'Employees must submit a leave request for approval.',
 'Company internet must be used for work-related tasks only.',
 'Company internet is a broadband internet.',
 'Employees can take an hour break.',
 'Interact with each employee with Respect']

In [ ]:
%pip install sentence-transformers

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
doc_vectors = model.encode(content_corpus)
doc_vectors

array([[ 0.02472513, -0.00908152,  0.03887125, ...,  0.01965648,
         0.04260009, -0.02707141],
       [ 0.03315507,  0.04853385,  0.04736267, ...,  0.10182001,
         0.09159282,  0.00358375],
       [-0.07135905, -0.03066469,  0.03183769, ..., -0.04109793,
         0.0652478 , -0.00688533],
       [-0.00383734, -0.02336758,  0.02958668, ..., -0.0441529 ,
         0.12559094, -0.03139849],
       [-0.01790442,  0.01495853,  0.08163832, ..., -0.03217229,
        -0.00513651,  0.05279532],
       [-0.00240894,  0.03361148, -0.06162645, ...,  0.04830879,
         0.03707639, -0.01683055]], shape=(6, 384), dtype=float32)

In [6]:
query = "What’s the internet usage policy?"
query_vec = model.encode([query])[0]
query_vec

array([ 2.39810487e-03, -4.11839783e-02, -2.52491049e-02, -4.67050187e-02,
        4.32101218e-03,  1.65876392e-02,  1.20891720e-01, -3.50352526e-02,
        2.16831663e-03, -1.62939774e-04,  2.62875874e-02,  9.05028135e-02,
       -2.66066510e-02, -1.82131901e-02,  3.06277778e-02,  1.67854782e-02,
        1.55614503e-02, -8.26497674e-02, -3.40456367e-02, -3.08671687e-02,
        7.89994970e-02, -3.16904262e-02,  1.35831339e-02,  9.12303512e-04,
       -1.05809728e-02,  3.91190760e-02, -3.48707139e-02,  8.63969108e-05,
       -3.52702588e-02,  3.56902294e-02,  9.55274422e-03, -3.57899107e-02,
        4.84949909e-03, -4.10226732e-02, -7.66861737e-02, -1.00646712e-01,
       -9.23561379e-02, -2.47272872e-03, -2.74321232e-02,  2.85045728e-02,
        2.86295936e-02, -7.78359175e-02, -2.46463087e-03,  9.98250470e-02,
        5.86107038e-02,  2.24836264e-02,  1.56639726e-04,  1.44218206e-02,
        5.55925304e-04,  3.22429463e-02,  1.03300020e-01,  3.42919715e-02,
        3.94683741e-02,  

In [7]:
import numpy as np

similarities = model.similarity(query_vec, doc_vectors)

# Ensure it's a 1D numpy array
similarities = np.asarray(similarities).squeeze()
similarities

array([0.12655614, 0.09747533, 0.44755304, 0.47982502, 0.11316215,
       0.04429528], dtype=float32)

In [8]:
# Now get top 3
top_3_indices = np.argsort(similarities)[::-1][:3]
print(top_3_indices)
top_scores = similarities[top_3_indices]
top_scores

[3 2 0]


array([0.47982502, 0.44755304, 0.12655614], dtype=float32)

In [9]:
top_docs = [documents[i]['content'] for i in top_3_indices]
# documents = [
#     {"section": "Pay Policies", "content": "Employees are paid bi-weekly via direct deposit."},
#     {"section": "Leave of Absence", "content": "Employees must submit a leave request for approval."},
#     {"section": "Internet Use", "content": "Company internet must be used for work-related tasks only."},
#     {"section": "Break at Work", "content": "Employees can take an hour break."},
#     {"section": "Harassment", "content": "Interact with each employee with Respect"}
# ]

print (top_docs)
context = "\n---\n".join(top_docs)
context

['Company internet is a broadband internet.', 'Company internet must be used for work-related tasks only.', 'Employees are paid bi-weekly via direct deposit.']


'Company internet is a broadband internet.\n---\nCompany internet must be used for work-related tasks only.\n---\nEmployees are paid bi-weekly via direct deposit.'

In [10]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env")
my_api_key = os.getenv("OPEN_AI_API_KEY")

my_client = OpenAI(api_key=my_api_key)
# my_client

def ask_question_open_ai(prompt):

    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create

    llm_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        # messages=[
        #     {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
        #     {"role": "user", "content": prompt}
        # ]
        messages=[
            {"role": "system", "content": '''
             You are an assistant who answers only based on the given context.
             '''},
            {"role": "user", "content": f"Context: {context}\n\n User Question: {query}"} 
        ]

    )
    return llm_response.choices[0].message.content  

In [11]:
print (query)
response = ask_question_open_ai(query)

What’s the internet usage policy?


In [12]:
print(f"User query: {query}")
print(f"Context: {context}")

print(f"\n\nOpen AI Response: {response}")

User query: What’s the internet usage policy?
Context: Company internet is a broadband internet.
---
Company internet must be used for work-related tasks only.
---
Employees are paid bi-weekly via direct deposit.


Open AI Response: - The company internet is broadband.
- It must be used for work-related tasks only (personal use is not permitted).
